# Building the Streamlit UI and Real-Time Simulation

This notebook creates a Streamlit dashboard for real-time UPI fraud detection.

The dashboard loads the trained Isolation Forest model, feature scaler, and tuned threshold. A user can enter transaction details, and the app returns an anomaly score plus a fraud-risk action.

## 1. Imports and Project Paths

In [6]:
from __future__ import annotations

from pathlib import Path
from textwrap import dedent


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
APP_DIR = PROJECT_ROOT / "app"
APP_PATH = APP_DIR / "streamlit_app.py"
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

APP_DIR.mkdir(parents=True, exist_ok=True)

## 2. Validate Required Artifacts

In [7]:
REQUIRED_ARTIFACTS = [
    MODELS_DIR / "isolation_forest_model.pkl",
    MODELS_DIR / "feature_scaler.pkl",
    MODELS_DIR / "threshold_metadata.pkl",
    PROCESSED_DATA_DIR / "upi_transactions_with_features.csv",
]


def validate_required_artifacts(paths: list[Path]) -> None:
    """Verify that the dashboard can load every required pipeline artifact."""
    missing_paths = [path for path in paths if not path.exists()]
    if missing_paths:
        formatted_paths = "\n".join(str(path) for path in missing_paths)
        raise FileNotFoundError(f"Missing required artifacts:\n{formatted_paths}")


validate_required_artifacts(REQUIRED_ARTIFACTS)
print("All dashboard artifacts are available.")

All dashboard artifacts are available.


## 3. Streamlit App Template

In [8]:
def build_streamlit_app_source() -> str:
    """Return the Streamlit dashboard source code."""
    return dedent(
        '''
        from __future__ import annotations

        from pathlib import Path
        from typing import Any

        import joblib
        import numpy as np
        import pandas as pd
        import streamlit as st


        PROJECT_ROOT = Path(__file__).resolve().parents[1]
        MODELS_DIR = PROJECT_ROOT / "models"
        PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

        MODEL_PATH = MODELS_DIR / "isolation_forest_model.pkl"
        SCALER_PATH = MODELS_DIR / "feature_scaler.pkl"
        THRESHOLD_METADATA_PATH = MODELS_DIR / "threshold_metadata.pkl"
        FEATURE_DATA_PATH = PROCESSED_DATA_DIR / "upi_transactions_with_features.csv"

        FEATURE_COLUMNS = [
            "hour_of_day",
            "day_of_week",
            "transaction_value",
            "is_first_time_receiver",
            "velocity_last_10min",
            "amount_vs_user_avg",
        ]

        DAY_NAME_TO_NUMBER = {
            "Monday": 0,
            "Tuesday": 1,
            "Wednesday": 2,
            "Thursday": 3,
            "Friday": 4,
            "Saturday": 5,
            "Sunday": 6,
        }


        @st.cache_resource
        def load_model_artifacts() -> tuple[Any, Any, dict[str, Any]]:
            """Load trained model, scaler, and threshold metadata once per app session."""
            model = joblib.load(MODEL_PATH)
            scaler = joblib.load(SCALER_PATH)
            threshold_metadata = joblib.load(THRESHOLD_METADATA_PATH)
            return model, scaler, threshold_metadata


        @st.cache_data
        def load_reference_transactions() -> pd.DataFrame:
            """Load engineered transactions for dashboard context and recent-risk examples."""
            if not FEATURE_DATA_PATH.exists():
                return pd.DataFrame()
            return pd.read_csv(FEATURE_DATA_PATH, parse_dates=["timestamp"])


        def build_feature_frame(
            amount: float,
            hour_of_day: int,
            day_name: str,
            receiver_type: str,
            velocity_last_10min: int,
            user_average_amount: float,
        ) -> pd.DataFrame:
            """Convert dashboard inputs into the model's six-feature schema."""
            safe_average = max(user_average_amount, 1.0)
            amount_vs_user_avg = amount / safe_average
            is_first_time_receiver = int(receiver_type == "New Beneficiary")

            return pd.DataFrame(
                [
                    {
                        "hour_of_day": hour_of_day,
                        "day_of_week": DAY_NAME_TO_NUMBER[day_name],
                        "transaction_value": amount,
                        "is_first_time_receiver": is_first_time_receiver,
                        "velocity_last_10min": velocity_last_10min,
                        "amount_vs_user_avg": amount_vs_user_avg,
                    }
                ],
                columns=FEATURE_COLUMNS,
            )


        def score_transaction(feature_frame: pd.DataFrame, model: Any, scaler: Any, threshold: float) -> dict[str, Any]:
            """Score a single transaction and return decision metadata."""
            scaled_features = scaler.transform(feature_frame)
            decision_score = float(model.decision_function(scaled_features)[0])
            anomaly_score = -decision_score
            is_suspicious = anomaly_score >= threshold

            return {
                "decision_score": decision_score,
                "anomaly_score": anomaly_score,
                "threshold": threshold,
                "is_suspicious": is_suspicious,
            }


        def render_result(score: dict[str, Any]) -> None:
            """Render the model decision in the dashboard."""
            if score["is_suspicious"]:
                st.error("High fraud risk: review or block this transaction.")
            else:
                st.success("Low fraud risk: transaction behavior looks normal.")

            metric_columns = st.columns(3)
            metric_columns[0].metric("Anomaly score", f"{score['anomaly_score']:.4f}")
            metric_columns[1].metric("Alert threshold", f"{score['threshold']:.4f}")
            metric_columns[2].metric("Decision", "Suspicious" if score["is_suspicious"] else "Normal")


        def render_reference_table(reference_data: pd.DataFrame) -> None:
            """Show recent synthetic transactions for context."""
            if reference_data.empty:
                return

            display_columns = [
                "timestamp",
                "sender_account_id",
                "receiver_account_id",
                "amount",
                "transaction_type",
                "is_fraud",
                "hour_of_day",
                "velocity_last_10min",
                "amount_vs_user_avg",
            ]
            available_columns = [column for column in display_columns if column in reference_data.columns]
            st.dataframe(reference_data[available_columns].tail(25), use_container_width=True)


        def main() -> None:
            """Run the Streamlit app."""
            st.set_page_config(page_title="UPI Fraud Detector", layout="wide")
            st.title("Real-Time UPI Fraud Detection")

            model, scaler, threshold_metadata = load_model_artifacts()
            reference_data = load_reference_transactions()
            threshold = float(threshold_metadata["best_threshold"])

            with st.sidebar:
                st.header("Transaction Input")
                amount = st.number_input("Transaction amount", min_value=1.0, max_value=500000.0, value=48000.0, step=500.0)
                user_average_amount = st.number_input("User average amount", min_value=1.0, max_value=250000.0, value=4000.0, step=500.0)
                hour_of_day = st.slider("Hour of day", min_value=0, max_value=23, value=2)
                day_name = st.selectbox("Day of week", list(DAY_NAME_TO_NUMBER.keys()), index=4)
                receiver_type = st.selectbox("Receiver type", ["New Beneficiary", "Known Beneficiary"])
                velocity_last_10min = st.slider("Transactions in last 10 minutes", min_value=1, max_value=25, value=4)

            feature_frame = build_feature_frame(
                amount=amount,
                hour_of_day=hour_of_day,
                day_name=day_name,
                receiver_type=receiver_type,
                velocity_last_10min=velocity_last_10min,
                user_average_amount=user_average_amount,
            )
            score = score_transaction(feature_frame, model, scaler, threshold)

            render_result(score)

            st.subheader("Model Features")
            st.dataframe(feature_frame, use_container_width=True)

            st.subheader("Tuned Threshold Metadata")
            st.json(threshold_metadata)

            st.subheader("Recent Simulated Transactions")
            render_reference_table(reference_data)


        if __name__ == "__main__":
            main()
        '''
    ).strip() + "\n"


streamlit_app_source = build_streamlit_app_source()
print(streamlit_app_source[:1200])

from __future__ import annotations

from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import streamlit as st


PROJECT_ROOT = Path(__file__).resolve().parents[1]
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

MODEL_PATH = MODELS_DIR / "isolation_forest_model.pkl"
SCALER_PATH = MODELS_DIR / "feature_scaler.pkl"
THRESHOLD_METADATA_PATH = MODELS_DIR / "threshold_metadata.pkl"
FEATURE_DATA_PATH = PROCESSED_DATA_DIR / "upi_transactions_with_features.csv"

FEATURE_COLUMNS = [
    "hour_of_day",
    "day_of_week",
    "transaction_value",
    "is_first_time_receiver",
    "velocity_last_10min",
    "amount_vs_user_avg",
]

DAY_NAME_TO_NUMBER = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6,
}


@st.cache_resource
def load_model_artifacts() -> tuple[Any, Any, dict[str, Any]]:
    """Load trained model, scaler, and t

## 4. Write the Streamlit App

In [9]:
def write_streamlit_app(app_path: Path, source_code: str) -> None:
    """Write the dashboard source file."""
    app_path.parent.mkdir(parents=True, exist_ok=True)
    app_path.write_text(source_code, encoding="utf-8")


write_streamlit_app(APP_PATH, streamlit_app_source)
print(f"Streamlit app written to: {APP_PATH}")

Streamlit app written to: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\app\streamlit_app.py


## 5. Validate the App Code

In [10]:
def validate_python_source(file_path: Path) -> None:
    """Compile the generated Python file to catch syntax errors early."""
    source = file_path.read_text(encoding="utf-8")
    compile(source, str(file_path), "exec")


validate_python_source(APP_PATH)
print("Streamlit app syntax validation passed.")

Streamlit app syntax validation passed.


## 6. Run Command

Run this command from the project root:

```bash
streamlit run app/streamlit_app.py
```